In [1]:
from datasets import Dataset

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re


c:\Users\agsto\Desktop\CMPE-252-summary-project\summary_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import BartTokenizer, BartForConditionalGeneration, Seq2SeqTrainer,DataCollatorForSeq2Seq, Seq2SeqTrainingArguments
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())       # True if GPU is usable
print("CUDA version:", torch.version.cuda)                # CUDA version PyTorch was built with
print("Number of GPUs:", torch.cuda.device_count())       # Number of GPUs detected
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu126
CUDA available: True
CUDA version: 12.6
Number of GPUs: 1
GPU name: NVIDIA GeForce RTX 4070 SUPER


## Importing and cleaning data set

In [3]:
documention_path_0 = "document_train_0.parquet" #file path
document_0 = pd.read_parquet(documention_path_0)

documention_path_1 = "document_train_1.parquet"
document_1 = pd.read_parquet(documention_path_1)

documention_path_2 = "document_train_2.parquet"
document_2 = pd.read_parquet(documention_path_2)

documention_path_3 = "document_train_3.parquet"
document_3 = pd.read_parquet(documention_path_3)

In [4]:
document = pd.concat([document_0, document_1, document_2, document_3])

In [5]:
document

,article,abstract
0,additive models @xcite provide an important fa...,additive models play an important role in semi...
1,the leptonic decays of a charged pseudoscalar ...,"we have studied the leptonic decay @xmath0 , v..."
2,the transport properties of nonlinear non - eq...,"in 84 , 258 ( 2000 ) , mateos conjectured that..."
3,studies of laser beams propagating through tur...,the effect of a random phase diffuser on fluct...
4,the so - called `` nucleon spin crisis '' rais...,with a special intention of clarifying the und...
...,...,...
13531,there has been a recent upsurge in interest in...,we present measurements of the integrated flux...
13532,the last few years have seen growing evidence ...,we show how a very accurate measurement of the...
13533,"the observed initial mass function of stars , ...",we demonstrate the feasibility of detecting di...
13534,numerical simulations of the growth of large s...,a clustering analysis is performed on two samp...


In [233]:
def clean_document(document):
    document['abstract'] = document['abstract'].str.replace(' \n', '', regex=False) #remove \n
    document['abstract'] = document['abstract'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
    document['abstract'] = document['abstract'].str.replace('  ', ' ', regex=False) #replace double space with single space

    document['article'] = document['article'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
    document['article'] = document['article'].str.replace('  ', ' ', regex=False) #replace double space with single space

    article_summary = document['article'].str.len().describe()
    abstract_summary = document['abstract'].str.len().describe()

    document = document[document['article'].str.len() >= article_summary['25%']] # has to be at least 25% article length
    document = document[document['article'].str.len() <= article_summary['75%']] # has to be at most 75% article length

    document = document[document['abstract'].str.len() >= abstract_summary['25%']] 
    document = document[document['abstract'].str.len() <= abstract_summary['75%']]

    document = document[document['article'].str.len() >= article_summary['25%']] 
    document = document[document['article'].str.len() <= article_summary['75%']]  

    document = document[document['abstract'].str.len() >= abstract_summary['25%']]
    document = document[document['abstract'].str.len() <= abstract_summary['75%']]

    document = document.drop_duplicates()
    document = document.reset_index(drop=True)
    
    return document

In [7]:
document = clean_document(document)

## Validation and Test cleaning

In [8]:
model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name, output_attentions=True, torch_dtype = torch.bfloat16)

Please make sure the generation config includes `forced_bos_token_id=0`. 
The following generation flags are not valid and may be ignored: ['output_attentions']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 667.97it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


In [4]:
validation_path = "document_validation.parquet" #file path
validation = pd.read_parquet(validation_path)

test_path = "document_test.parquet"
test = pd.read_parquet(test_path)

In [5]:
validation = clean_document(validation)

In [35]:
test = clean_document(test)

## Training

In [9]:
train_dataset = Dataset.from_pandas(document)
validation_dataset   = Dataset.from_pandas(validation)
test_dataset  = Dataset.from_pandas(test)

NameError: name 'validation' is not defined

In [ ]:
label = tokenizer(
    list(train_dataset["abstract"]),
    truncation=True,
    max_length=1024,
    padding=False
)

# Get lengths
label_lengths = [len(ids) for ids in label["input_ids"]]

In [ ]:
abstract_tokens_summary = pd.DataFrame(label_lengths).describe()[0]
abstract_tokens_summary

count    15490.000000
mean       193.410781
std         42.827134
min        106.000000
25%        159.000000
50%        189.000000
75%        223.000000
max        544.000000
Name: 0, dtype: float64

In [ ]:
token_len_IQR =  abstract_tokens_summary['75%'] - abstract_tokens_summary['25%']
1.5 * token_len_IQR

np.float64(96.0)

In [ ]:
abstract_tokens_summary['25%'] - (1.5 * token_len_IQR)

np.float64(63.0)

In [ ]:
abstract_tokens_summary['75%'] + (1.5 * token_len_IQR)

#pad to 320 should be a clean number

np.float64(319.0)

In [ ]:
model = BartForConditionalGeneration.from_pretrained(r"bart_finetuned\checkpoint-7746").cuda()
tokenizer = BartTokenizer.from_pretrained(r"bart_finetuned\checkpoint-7746")

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 1132.74it/s, Materializing param=model.shared.weight]                                   


In [ ]:
def tokenize_document(dataset):
    model_input = tokenizer(dataset["article"], max_length = 1024, truncation = True)
    labels = tokenizer(dataset["abstract"], max_length = 320, truncation = True)

    model_input["labels"] = labels['input_ids']
    return model_input

In [ ]:
tokenized_dataset = train_dataset.map(tokenize_document, batched=True)
validation_tokenized = validation_dataset.map(tokenize_document, batched=True, remove_columns=validation_dataset.column_names)
test_tokenized = test_dataset.map(tokenize_document, batched=True, remove_columns=test_dataset.column_names)

Map: 100%|██████████| 1715/1715 [00:05<00:00, 298.92 examples/s]


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest"  # pads to the longest sequence in each batch
)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_finetuned",      # folder to save checkpoints
    num_train_epochs=3,                  # number of passes through the dataset
    per_device_train_batch_size=4,       # safe for 12GB VRAM
    per_device_eval_batch_size=4,        # same as train batch size
    learning_rate=2e-5,                  # learning rate
    weight_decay=0.01,                   # regularization
    save_total_limit=3,                   # keep last 3 checkpoints
    eval_strategy="epoch",          # evaluate once per epoch
    save_strategy="epoch",                # save checkpoint once per epoch
    logging_strategy="steps",             # log every N steps
    logging_steps=100,                    # adjust if needed
    bf16=True,                            # mixed precision for VRAM efficiency
    predict_with_generate=True            # necessary for seq2seq evaluation
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=validation_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train(resume_from_checkpoint="./bart_finetuned/checkpoint-7746") #pick up from Epoch = 2

### Epoch	Training Loss	Validation Loss
### 1	2.795941	2.686332
### 2	2.850331	2.683255
### 3	2.777663	2.682559

## ROUGE

In [ ]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # If model returns tuple
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Decode predictions
    decoded_preds = tokenizer.batch_decode(
        predictions, skip_special_tokens=True
    )

    # Replace -100 in labels (ignore index) so they can be decoded
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(
        labels, skip_special_tokens=True
    )

    # Strip whitespace
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"]
    }

## Trained model ROUGE

In [ ]:
#epoch = 3

rouge = evaluate.load("rouge")

predictions = trainer.predict(validation_tokenized)

preds = predictions.predictions
labels = predictions.label_ids

labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

results = rouge.compute(
    predictions=decoded_preds,
    references=decoded_labels,
    use_stemmer=True
)

print({k: round(v * 100, 4) for k, v in results.items()})

{'rouge1': np.float64(44.6801), 'rouge2': np.float64(15.9898), 'rougeL': np.float64(24.713), 'rougeLsum': np.float64(24.7124)}


### ROUGE results from epoch = 2

{'rouge1': np.float64(44.6586), 'rouge2': np.float64(15.9589), 'rougeL': np.float64(24.7595), 'rougeLsum': np.float64(24.7636)}

### ROUGE results from epoch = 3

{'rouge1': np.float64(44.6801), 'rouge2': np.float64(15.9898), 'rougeL': np.float64(24.713), 'rougeLsum': np.float64(24.7124)}

# Summary Generation comparison

In [225]:
train_section = pd.read_parquet('section_train_4.parquet')

train_section['article'] = train_section['article'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
train_section['article'] = train_section['article'].str.replace('  ', ' ', regex=False) #replace double space with single space

train_document = pd.read_parquet('document_train_4.parquet')

train_document['article'] = train_document['article'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
train_document['article'] = train_document['article'].str.replace('  ', ' ', regex=False) #replace double space with single space

In [191]:
test_ind = 100

In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BartForConditionalGeneration.from_pretrained(r"bart_finetuned\checkpoint-7746", attn_implementation="eager").to(device)
tokenizer = BartTokenizer.from_pretrained(r"bart_finetuned\checkpoint-7746")

inputs = tokenizer(
    train_document['article'][test_ind],
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=1024   # BART max input length
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    summary_ids = model.generate(
        **inputs,
        max_length=320,
        min_length=30,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 1098.80it/s, Materializing param=model.shared.weight]                                   


we present a series of controlled three - dimensional hydrodynamic simulations of single type ii supernova remnant remnants of neutron - star binary mergers ( nsbms ) . the ejected mass and initial structure of the tidal tails are calculated using three -dimensional smoothed particle hydrodynamics simulations of the merger event . the resultant homologous structure is then mapped into an adaptive mesh refinement ( amr ) simulation with optically - thin radiative cooling . the results are used to construct sub - resolution prescriptions for galaxy - scale simulations with inadequate resolution to properly define the the cooling radius of nsbm blast waves . our simulations of isolated nSBms are also used to calculate the temperature variation within the expanding , cooling layer layer layer of the remnant . once the cooling function is calculated , the unbound tidal tails we calculate are similar to those of type ii and type i supernovae . we find that the abundance patterns of -process 

In [228]:
train_document['abstract'][test_ind]

'the @xmath0-process nuclei are robustly synthesized in the material ejected during a neutron star binary merger ( nsbm ) , as tidal torques transport angular momentum and energy through the outer lagrange point in the form of a vast tidal tail . if nsbm are indeed solely responsible for the solar system @xmath0-process abundances , a galaxy like our own would require to host a few nsbm per million years , with each event ejecting , on average , about @xmath1 of @xmath0-process material . because the ejecta velocities in the tidal tail are significantly larger than in ordinary supernovae , nsbm deposit a comparable amount of energy into the interstellar medium ( ism ) . \n in contrast to extensive efforts studying spherical models for supernova remnant evolution , calculations quantifying the impact of nsbm ejecta in the ism have been lacking . to better understand their evolution in a cosmological context \n , we perform a suite of three - dimensional hydrodynamic simulations with opt

In [229]:
train_document['article'][test_ind]

'the specific physical conditions and nuclear physics pathways required for the -process were originally identified in the pioneering work by . but the particular astrophysical site remains open to more than one interpretation . early work on the subject identified both type ii supernovae and neutron - star binary mergers ( nsbms ; * ? ? ? * ; * ? ? ? * ) as likely candidate events to hold r - process . nsbms are significantly much rarer than type ii supernovae and take place far from their birth places , reaching distance up to a few mpc from their host halo ( e.g. * ? ? ? the two mechanisms synthesize different quantities of -process material , about and for each type ii supernovae and nsbm , respectively ( e.g. * ? ? ? . these differences should give rise to clear signatures in the enrichment pattern of -process elements in galaxies and may ultimately help constrain the dominant production mechanism .  with many difficulties getting the necessary conditions to produce the -process i

# Cross Attention

In [253]:
def get_token_score_pairs(article, path = r"bart_finetuned\checkpoint-7746"):
    """"
    This function gives the importance score of each token. 
    Be mindful and use sectioned article with \n as we will use that to detect new lines

    Parameters
    ----------
    article: str
        the article we want to get the a summary from
    
    path: path
        path to the folder of pre trained model

    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #prioritze cuda
    model = BartForConditionalGeneration.from_pretrained(path, attn_implementation="eager").to(device)
    tokenizer = BartTokenizer.from_pretrained(path)

    inputs = tokenizer(
        article,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=1024
    )

    inputs = {k: v.to(device) for k, v in inputs.items()} #makes all inputs set to cuda if possible otherwise cpu

    outputs = model(
        **inputs,
        output_attentions=True,
        return_dict=True
    )

    cross_attentions = outputs.cross_attentions #gets the cross attention scores for each token in the article
    last_layer = cross_attentions[-1] #looks at last layer
    avg_heads = last_layer.mean(dim=1) #gets the average of all heads of attention scores for each token 
    token_importance = avg_heads.mean(dim=1) #gives overall importance of each token

    input_ids = inputs["input_ids"][0]
    tokens = tokenizer.convert_ids_to_tokens(input_ids) #gives the tokens
    
    token_scores = [(token, float(score.detach())) for token, score in zip(tokens, token_importance[0])] #gives the list of tuples with tokens and importance score

    return token_scores
    

In [254]:
def rank_important_lines(token_scores):
    """"
    This function ranks lines based on their importance.
    This only works if we use a sectioned article with \n as we will use that to detect new lines

    Parameters
    ----------
    token_scores: list of tuples (token, score)
        token_scores is a list of tuples where each tuple contains a token and its corresponding importance score

    """
    lines = []
    current_line = []
    current_score = 0

    for token, score in token_scores:
        if token == "Ċ": #"Ċ" represents the start of new line so
            if current_line:
                lines.append((" ".join(current_line).replace("Ġ","").strip(' . '), current_score)) #replace Ġ token with space and removed period
            current_line = [] 
            current_score = 0
        else:
            current_line.append(token) #add word/symbol to the current line
            current_score += score #add score that belongs to the line
    if current_line: #for the last line
        lines.append((" ".join(current_line).replace("Ġ","").strip(' . '), current_score)) #replace Ġ token with space and removed period

    lines[0]  = (lines[0][0].replace("<s>", "").strip(),  lines[0][1]) #take out starting token
    lines[-1] = (lines[-1][0].replace("</s>", "").strip(), lines[-1][1]) #take out ending token

    ranked_lines = sorted(lines, key=lambda x: x[1], reverse=True) #ranks the lines based on the score in desc order

    return ranked_lines

In [255]:
pairs = get_token_score_pairs(train_section['article'][2])
rank_important_lines(pairs)

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 1193.47it/s, Materializing param=model.shared.weight]                                   


[('long before massive black holes were accepted to be present in the centers of galaxies , argued that a binary black hole created in the merger of two galaxies would eject stars from the center of the newly created system as the binary slowly hardened',
  0.3212294578552246),
 ("the b im od al distribution was initially inferred from fits to the surface phot ometry using the `` n uk er law '' ( ; see equation [ eq n : n uk er ] ) , which has the as ym pt otic form as however , demonstrated that non - param etric treatment of the surface phot ometry recovered an identical b im od al distribution , and further that b im od ality was a feature of the lumin osity _ density _ distributions of early type galaxies , not just the surface brightness distributions",
  0.0665128231048584),
 ('the b im od al c usp distribution would thus be one key to understanding the central structure of ellipt ical galaxies over a history of mer gers .  as larger samples of ellipt ical galaxies were observed 

In [256]:
print(summary)

we present a series of controlled three - dimensional hydrodynamic simulations of single type ii supernova remnant remnants of neutron - star binary mergers ( nsbms ) . the ejected mass and initial structure of the tidal tails are calculated using three -dimensional smoothed particle hydrodynamics simulations of the merger event . the resultant homologous structure is then mapped into an adaptive mesh refinement ( amr ) simulation with optically - thin radiative cooling . the results are used to construct sub - resolution prescriptions for galaxy - scale simulations with inadequate resolution to properly define the the cooling radius of nsbm blast waves . our simulations of isolated nSBms are also used to calculate the temperature variation within the expanding , cooling layer layer layer of the remnant . once the cooling function is calculated , the unbound tidal tails we calculate are similar to those of type ii and type i supernovae . we find that the abundance patterns of -process 

In [ ]:
train_document['abstract'][test_ind]

'an abelian cover is a finite morphism @xmath0 of varieties which is the quotient map for a generically faithful action of a finite abelian group @xmath1 . \n abelian covers with @xmath2 smooth and @xmath3 normal were studied in @xcite .    here \n we study the non - normal case , assuming that @xmath3 and @xmath2 are @xmath4 varieties that have at worst normal crossings outside a subset of codimension @xmath5 . \n special attention is paid to the case of @xmath6-covers of surfaces , which is used in @xcite to construct explicitly compactifications of some components of the moduli space of surfaces of general type .'

In [ ]:
train_document['article'][test_ind]

"an abelian cover is a finite morphism of varieties which is the quotient map for a generically faithful action of a finite abelian group . this means that for every component of the -action on the restricted cover is faithful . the paper contains a comprehensive theory of such covers in the case when is smooth and is normal . the covers are described in terms of the _ building data _ consisting of branch divisors ranging over cyclic subgroups , and line bundles with ranging over the character group of . this collection must satisfy the _ fundamental relations_.  here , we extend this theory to the case of singular varieties . namely , we allow and to be varieties satisfying serre s condition and having double crossing singularities in codimension 1 , which we abbreviate to g.d.c . for `` generically double crossings '' ( see [ ssec : ysmooth ] for the precise definition ) . our interest in this case lies in applications to the moduli theory . such non - normal abelian covers appear in